# Human Activity Recognition
I compare PCA and LDA representations with SVC classifiers using the official UCI HAR train/test split. Scaling and dimensionality reduction are fitted on training data only. Set HAR_DATASET_DIR to the extracted dataset directory.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
path = Path(os.environ.get("HAR_DATASET_DIR", "data/UCI HAR Dataset"))
X_train = pd.read_csv(path / "train/X_train.txt", sep=r"\s+", header=None)
y_train = pd.read_csv(path / "train/y_train.txt", header=None)
X_test = pd.read_csv(path / "test/X_test.txt", sep=r"\s+", header=None)
y_test = pd.read_csv(path / "test/y_test.txt", header=None)

In [ ]:
print("Training:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
y_train_values = y_train.values.ravel()
y_test_values = y_test.values.ravel()

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
pca = PCA(n_components=0.90)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
print("PCA components for 90% variance:", pca.n_components_)

In [ ]:
print("Retained variance:", cumulative_variance[-1])

In [ ]:
plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train_values, s=5)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Training data: PCA"); plt.show()

In [ ]:
lda = LDA()
X_train_lda = lda.fit_transform(X_train_scaled, y_train_values)
X_test_lda = lda.transform(X_test_scaled)

In [ ]:
plt.scatter(X_train_lda[:, 0], X_train_lda[:, 1], c=y_train_values, s=5)
plt.xlabel("LD1"); plt.ylabel("LD2"); plt.title("Training data: LDA"); plt.show()

In [ ]:
print("Official training and test partitions are kept separate.")

In [ ]:
model = SVC()
model.fit(X_train_pca, y_train_values)
y_pred = model.predict(X_test_pca)

In [ ]:
def evaluate(y_true, predicted):
    return {"accuracy": accuracy_score(y_true, predicted),
            "precision": precision_score(y_true, predicted, average="weighted", zero_division=0),
            "recall": recall_score(y_true, predicted, average="weighted", zero_division=0),
            "f1": f1_score(y_true, predicted, average="weighted", zero_division=0)}
print("PCA + SVC:", evaluate(y_test_values, y_pred))

In [ ]:
model_lda = SVC()
model_lda.fit(X_train_lda, y_train_values)
y_pred_lda = model_lda.predict(X_test_lda)
print("LDA + SVC:", evaluate(y_test_values, y_pred_lda))